In [1]:
import pandas as pd
import psycopg2
from sqlalchemy import create_engine
from psycopg2 import sql
import os
import time

In [2]:


# Cell 2: PostgreSQL Connection (Update your credentials)
DB_HOST = 'localhost'
DB_PORT = 5433
DB_NAME = 'fo_db'
DB_USER = 'postgres'  # From your earlier query
DB_PASS = 'Lucky1234'

In [3]:
engine = create_engine(f'postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}')
conn = psycopg2.connect(
    host=DB_HOST, port=DB_PORT, dbname=DB_NAME, 
    user=DB_USER, password=DB_PASS
)
cur = conn.cursor()

print("Connected to PostgreSQL fo_db")


Connected to PostgreSQL fo_db


In [4]:
# Cell 3: Load CSV (Your local file)
csv_path = r"C:\Users\tilak\Downloads\Qodedataset\3mfanddo.csv"
df = pd.read_csv(csv_path)
print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")
print(df.head())
print(df.dtypes)

Loaded 2,533,210 rows, 16 columns
   Unnamed: 0 INSTRUMENT     SYMBOL    EXPIRY_DT  STRIKE_PR OPTION_TYP  \
0      160393     FUTIDX  BANKNIFTY  29-Aug-2019        0.0         XX   
1      160394     FUTIDX  BANKNIFTY  26-Sep-2019        0.0         XX   
2      160395     FUTIDX  BANKNIFTY  31-Oct-2019        0.0         XX   
3      160396     FUTIDX      NIFTY  29-Aug-2019        0.0         XX   
4      160397     FUTIDX      NIFTY  26-Sep-2019        0.0         XX   

       OPEN      HIGH       LOW     CLOSE  SETTLE_PR  CONTRACTS  VAL_INLAKH  \
0  28805.65  28924.00  28140.55  28499.30   28499.30   214569.0  1225914.96   
1  28926.40  29030.55  28251.70  28611.45   28611.45     2484.0    14245.95   
2  29000.00  29105.00  28355.55  28699.05   28699.05      598.0     3434.43   
3  11098.40  11098.40  10901.10  11015.35   11015.35   199881.0  1650955.24   
4  11136.35  11145.20  10955.00  11066.60   11066.60     5283.0    43841.57   

     OPEN_INT  CHG_IN_OI    TIMESTAMP  
0   16

In [5]:
# Cell 4: Add Exchange Column (NSE for Kaggle data)
df['EXCHANGE'] = 'NSE'
print("Added exchange column")


Added exchange column


In [6]:
# Cell 5: Create Staging Table with uppercase column names
create_staging_sql = """
DROP TABLE IF EXISTS stg_fo_raw;
CREATE TABLE stg_fo_raw (
    "ROW_ID" BIGINT,
    "INSTRUMENT" TEXT,
    "SYMBOL" TEXT,
    "EXPIRY_DT" VARCHAR(20),
    "STRIKE_PR" NUMERIC(12,4),
    "OPTION_TYP" TEXT,
    "OPEN" NUMERIC(12,4),
    "HIGH" NUMERIC(12,4),
    "LOW" NUMERIC(12,4),
    "CLOSE" NUMERIC(12,4),
    "SETTLE_PR" NUMERIC(12,4),
    "CONTRACTS" NUMERIC(15,4),
    "VAL_INLAKH" NUMERIC(12,4),
    "OPEN_INT" NUMERIC(15,0),
    "CHG_IN_OI" NUMERIC(15,0),
    "TIMESTAMP" VARCHAR(20),
    "EXCHANGE" VARCHAR(10)
);
"""

cur.execute(create_staging_sql)
conn.commit()
print("Staging table created")




Staging table created


In [7]:
print(df.columns)

Index(['Unnamed: 0', 'INSTRUMENT', 'SYMBOL', 'EXPIRY_DT', 'STRIKE_PR',
       'OPTION_TYP', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'SETTLE_PR', 'CONTRACTS',
       'VAL_INLAKH', 'OPEN_INT', 'CHG_IN_OI', 'TIMESTAMP', 'EXCHANGE'],
      dtype='object')


In [8]:
# If Unnamed: 0 exists, rename it
df.columns = [c.upper() for c in df.columns]
if 'UNNAMED: 0' in df.columns:
    df.rename(columns={'UNNAMED: 0': 'ROW_ID'}, inplace=True)
else:
    # Otherwise create a row_id column
    df.insert(0, 'ROW_ID', range(1, len(df)+1))

In [9]:
df['EXPIRY_DT'] = pd.to_datetime(df['EXPIRY_DT'], format='%d-%b-%y', errors='coerce')
df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'], format='%d-%b-%y', errors='coerce')


In [10]:
invalid_expiry = df[df['EXPIRY_DT'].isna()]
invalid_ts = df[df['TIMESTAMP'].isna()]

print(f"Invalid expiry dates:\n{invalid_expiry}")
print(f"Invalid timestamps:\n{invalid_ts}")


Invalid expiry dates:
           ROW_ID INSTRUMENT     SYMBOL EXPIRY_DT  STRIKE_PR OPTION_TYP  \
0          160393     FUTIDX  BANKNIFTY       NaT        0.0         XX   
1          160394     FUTIDX  BANKNIFTY       NaT        0.0         XX   
2          160395     FUTIDX  BANKNIFTY       NaT        0.0         XX   
3          160396     FUTIDX      NIFTY       NaT        0.0         XX   
4          160397     FUTIDX      NIFTY       NaT        0.0         XX   
...           ...        ...        ...       ...        ...        ...   
2533205  32769933     OPTSTK       ZEEL       NaT      390.0         PE   
2533206  32769934     OPTSTK       ZEEL       NaT      400.0         PE   
2533207  32769935     OPTSTK       ZEEL       NaT      410.0         PE   
2533208  32769936     OPTSTK       ZEEL       NaT      420.0         PE   
2533209  32769937     OPTSTK       ZEEL       NaT      430.0         PE   

             OPEN      HIGH       LOW     CLOSE  SETTLE_PR  CONTRACTS  \
0   

In [11]:
df.to_sql(
    'stg_fo_raw',
    engine,
    if_exists='append',
    index=False,
    chunksize=50_000,
    method='multi'
)

print("Bulk loaded to stg_fo_raw")

Bulk loaded to stg_fo_raw


In [12]:
print("Waiting 20 minutes before closing connections...")
time.sleep(20 * 60)

Waiting 20 minutes before closing connections...


In [13]:
cur.execute("SELECT COUNT(*) FROM stg_fo_raw")
row_count = cur.fetchone()[0]
print(f"Import complete: {row_count:,} rows loaded")


Import complete: 2,533,210 rows loaded


In [ ]:
# Cell 10: Close connections
cur.close()
conn.close()
engine.dispose()
print("ETL Pipeline Complete!")